# Aula Consulta
ACA 1 Computación en la Nube · Grupo 8

ANTERNOL DE LOS REYES BEDOYA NARANJO

MIGUEL JOSÉ GUTIÉRREZ CASTRO

CARLOS IVAN NUÑEZ FIERRO

JUAN RICARDO SANDOVAL ALVAREZ

NATALIA SEPULVEDA RAMIREZ

La carpeta resultados conserva la ejecución real realizada en el portátil el 30 de octubre de 2026. Las celdas de conexión e indexación necesitan credenciales propias para volver a ejecutarse. Sus salidas no se han simulado.

In [ ]:
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-r','requirements.txt'])

## 1. Credenciales y conexión
Ejecute desde la carpeta del proyecto. La clave se lee desde .env local o se solicita de forma oculta y se carga como variable de entorno. No incluir .env en la entrega.

In [ ]:
from pathlib import Path
import os,getpass
from dotenv import load_dotenv
load_dotenv(Path.cwd()/'.env')
if not os.getenv('QDRANT_URL'):
    os.environ['QDRANT_URL']=input('Punto final HTTPS: ').strip()
if not os.getenv('QDRANT_API_KEY'):
    os.environ['QDRANT_API_KEY']=getpass.getpass('Clave API oculta: ')
from proyecto import *
preparar_carpetas()
conectar()
print('Conexión remota confirmada')

## 2. Dataset estructurado y SQL
395 registros portugueses. Consultas de resumen, apoyo familiar y tiempo de estudio. Los documentos CUN se vinculan por temas, sin relación individual ni equivalencia normativa.

In [ ]:
resumen=base_sql()
resumen

## 3. Ingesta
Tres PDF del portal CUN. Conservar archivo, URL, página y hash. Revisar texto extraído.

In [ ]:
paginas=cargar_paginas()
print('Páginas con texto:',len(paginas))
print(paginas[0]['texto'][:700])

## 4. Chunking
Hasta 220 palabras, solapamiento de 40 y límite de 460 tokens incluyendo prefijo. Respetar páginas. E5 multilingüe usa 384 dimensiones y prefijos passage: / query:. La guía permite modelos alternativos, por lo que se documenta E5 en lugar del ejemplo específico Gemini/RETRIEVAL_QUERY.

In [ ]:
m=modelo_local()
fragmentos=fragmentar(paginas,m.tokenizer)
print('Fragmentos:',len(fragmentos),'Máximo tokens:',max(x['tokens'] for x in fragmentos))
fragmentos[0]

## 5. Indexación remota
Distancia coseno, normalización, lotes de 32. IDs estables para el mismo corpus. No elimina colecciones. Cambios de corpus requieren colección nueva.

In [ ]:
reporte=indexar(m,fragmentos)
reporte

## 6. Top-3 y puntuaciones
Cuatro consultas educativas y una fuera de tema. Coseno recalculado con vectores. Latencia solo de consulta remota, excluye embeddings.

In [ ]:
df=validar(m)
df[['pregunta','rango','similitud','coseno_recalculado','archivo','pagina','latencia_remota_s']]

In [ ]:
for f in df.to_dict('records'):
    print('\nPregunta:',f['pregunta'],'Rango:',f['rango'],'Score:',round(f['similitud'],4))
    print(f['archivo'],'página',f['pagina'])
    print(f['texto'])

## 7. Evaluación humana
Lea cada fragmento y asigne 0/1 según su relevancia. La similitud no es probabilidad ni precisión. Documente que la consulta ajena carece de evidencia adecuada.

In [ ]:
revision_path=BASE/'resultados/revision_relevancia.csv'
if not revision_path.exists():
    revision=df.copy()
    revision['relevante']=None
    revision.to_csv(revision_path,index=False)
print('Complete relevante (0/1). Se conservan anotaciones anteriores.')

In [ ]:
import pandas as pd
r=pd.read_csv(BASE/'resultados/revision_relevancia.csv')
r=r[r['pregunta']!=PREGUNTAS[-1]]
if r['relevante'].isna().any():
    print('Pendiente revisión humana. Sin cálculo de precisión todavía.')
else:
    assert r['relevante'].isin([0,1]).all()
    display(r.groupby('pregunta')['relevante'].mean().rename('precision_at_3'))

## 8. Resultados y límites observados
La ejecución aportada registró 183 puntos de 384 dimensiones y 15 resultados para cinco preguntas. La latencia remota fue de 0,597 a 0,643 s. El primer resultado de derechos remite a la página 26 del PDF. Para apoyo económico, el segundo fragmento es más pertinente que el primero. La pregunta de pizza recuperó textos ajenos a la consulta. No se reporta precisión validada por el grupo. El siguiente bloque lee los archivos aportados, sin ejecutar consultas nuevas.

Fuentes: fuentes.json y https://doi.org/10.24432/C5TG7T. Modelo: https://huggingface.co/intfloat/multilingual-e5-small.

In [ ]:
import json, pandas as pd
guardados=json.loads((BASE/'resultados/consultas.json').read_text(encoding='utf-8'))
pd.DataFrame(guardados)[['pregunta','rango','similitud','archivo','pagina','latencia_remota_s']]